In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

In [2]:
X, y = make_classification(
    n_samples=200,
    n_features=5,
    n_informative=3,
    n_classes=2,
    weights=[0.9, 0.1],
    shuffle=True,
    random_state=7,
)

print("There are {} positive instances.".format(y.sum()))

There are 21 positive instances.


In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, stratify=y, random_state=7)

print("Train set labels distibution: {}".format(np.bincount(y_train)))
print("Test set labels distibution:  {}".format(np.bincount(y_test)))

Train set labels distibution: [120  14]
Test set labels distibution:  [59  7]


In [4]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test)

In [5]:
params = {"objective": "binary:logistic", "max_depth": 1, "eta": 1}
num_rounds = 15

In [6]:
xgb_model = xgb.train(params, dtrain, num_rounds)
y_test_preds = (xgb_model.predict(dtest) > 0.5).astype("int")

In [7]:
pd.crosstab(
    pd.Series(y_test, name="Actual"),
    pd.Series(y_test_preds, name="Predicted"),
    margins=True,
)

Predicted,0,1,All
Actual,,,
0,59,0,59
1,6,1,7
All,65,1,66


In [8]:
print("Accuracy: {0:.2f}".format(accuracy_score(y_test, y_test_preds)))
print("Precision: {0:.2f}".format(precision_score(y_test, y_test_preds)))
print("Recall: {0:.2f}".format(recall_score(y_test, y_test_preds)))

Accuracy: 0.91
Precision: 1.00
Recall: 0.14


In [9]:
weights = np.zeros(len(y_train))
weights[y_train == 0] = 1
weights[y_train == 1] = 5

dtrain = xgb.DMatrix(X_train, label=y_train, weight=weights)
dtest = xgb.DMatrix(X_test)

In [10]:
xgb_model = xgb.train(params, dtrain, num_rounds)
y_test_preds = (xgb_model.predict(dtest) > 0.5).astype("int")

In [11]:
pd.crosstab(
    pd.Series(y_test, name="Actual"),
    pd.Series(y_test_preds, name="Predicted"),
    margins=True,
)

Predicted,0,1,All
Actual,,,
0,58,1,59
1,5,2,7
All,63,3,66


In [12]:
print("Accuracy: {0:.2f}".format(accuracy_score(y_test, y_test_preds)))
print("Precision: {0:.2f}".format(precision_score(y_test, y_test_preds)))
print("Recall: {0:.2f}".format(recall_score(y_test, y_test_preds)))

Accuracy: 0.91
Precision: 0.67
Recall: 0.29


In [13]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test)

In [14]:
train_labels = dtrain.get_label()

ratio = float(np.sum(train_labels == 0)) / np.sum(train_labels == 1)
params["scale_pos_weight"] = ratio

In [15]:
xgb_model = xgb.train(params, dtrain, num_rounds)
y_test_preds = (xgb_model.predict(dtest) > 0.5).astype("int")

pd.crosstab(
    pd.Series(y_test, name="Actual"),
    pd.Series(y_test_preds, name="Predicted"),
    margins=True,
)

Predicted,0,1,All
Actual,,,
0,56,3,59
1,5,2,7
All,61,5,66


In [16]:
print("Accuracy: {0:.2f}".format(accuracy_score(y_test, y_test_preds)))
print("Precision: {0:.2f}".format(precision_score(y_test, y_test_preds)))
print("Recall: {0:.2f}".format(recall_score(y_test, y_test_preds)))

Accuracy: 0.88
Precision: 0.40
Recall: 0.29
